In [0]:
dbutils.widgets.text("env", "dev")
env = dbutils.widgets.get("env")

spark.sql(f"USE CATALOG {env}_lakehouse")


In [0]:
%sql
USE SCHEMA gold;

CREATE SCHEMA IF NOT EXISTS gold
COMMENT 'Gold layer - star schema for analytics';

-- dim_date
CREATE TABLE IF NOT EXISTS dim_date (
  date_key INT,
  trade_date DATE,
  year INT,
  month INT,
  day INT
)
USING DELTA
COMMENT 'Date dimension';

-- dim_stock (SCD Type 2)
CREATE TABLE IF NOT EXISTS dim_stock (
  stock_key INT,
  symbol STRING,
  effective_start_date DATE,
  effective_end_date DATE,
  is_current BOOLEAN
)
USING DELTA
COMMENT 'Stock dimension - SCD Type 2';

-- fact table
CREATE TABLE IF NOT EXISTS fact_stock_prices (
  stock_key INT,
  date_key INT,
  open DOUBLE,
  high DOUBLE,
  low DOUBLE,
  close DOUBLE,
  volume BIGINT
)
USING DELTA
COMMENT 'Fact table for daily stock prices';


In [0]:
%sql
USE SCHEMA gold;

MERGE INTO dim_date tgt
USING (
  SELECT
    date_key,
    trade_date,
    year,
    month,
    day
  FROM (
    SELECT
      CAST(date_format(trade_date, 'yyyyMMdd') AS INT) AS date_key,
      trade_date,
      year(trade_date)  AS year,
      month(trade_date) AS month,
      day(trade_date)   AS day,
      ROW_NUMBER() OVER (
        PARTITION BY trade_date
        ORDER BY trade_date
      ) AS rn
    FROM silver.stock_prices_cleaned
  )
  WHERE rn = 1
) src
ON tgt.trade_date = src.trade_date
WHEN NOT MATCHED THEN
  INSERT *
;



In [0]:
%sql
USE SCHEMA gold;

MERGE INTO dim_stock tgt
USING (
  SELECT DISTINCT symbol
  FROM silver.stock_prices_cleaned
) src
ON tgt.symbol = src.symbol
AND tgt.is_current = true
WHEN MATCHED THEN
  UPDATE SET
    tgt.is_current = true;


In [0]:
%sql
USE SCHEMA gold;

INSERT INTO dim_stock
SELECT
  (SELECT COALESCE(MAX(stock_key), 0) + 1 FROM dim_stock) AS stock_key,
  src.symbol,
  current_date() AS effective_start_date,
  DATE '9999-12-31' AS effective_end_date,
  true AS is_current
FROM (
  SELECT DISTINCT symbol
  FROM silver.stock_prices_cleaned
) src
LEFT JOIN dim_stock tgt
  ON src.symbol = tgt.symbol
 AND tgt.is_current = true
WHERE tgt.symbol IS NULL;


In [0]:
%sql
USE SCHEMA gold;

MERGE INTO fact_stock_prices tgt
USING (
  SELECT
    stock_key,
    date_key,
    open,
    high,
    low,
    close,
    volume
  FROM (
    SELECT
      st.stock_key,
      d.date_key,
      s.open,
      s.high,
      s.low,
      s.close,
      s.volume,
      ROW_NUMBER() OVER (
        PARTITION BY st.stock_key, d.date_key
        ORDER BY s.trade_date DESC
      ) AS rn
    FROM silver.stock_prices_cleaned s
    JOIN dim_date d
      ON s.trade_date = d.trade_date
    JOIN dim_stock st
      ON s.symbol = st.symbol
     AND st.is_current = true
  )
  WHERE rn = 1
) src
ON tgt.stock_key = src.stock_key
AND tgt.date_key = src.date_key
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *
;



In [0]:
%sql
SELECT
  st.symbol,
  dt.trade_date,
  f.open,
  f.close,
  f.volume
FROM fact_stock_prices f
JOIN dim_stock st
  ON f.stock_key = st.stock_key
JOIN dim_date dt
  ON f.date_key = dt.date_key
ORDER BY dt.trade_date DESC
LIMIT 10;


In [0]:
def assert_zero(query: str, check_name: str):
    count = spark.sql(query).collect()[0][0]
    if count != 0:
        raise Exception(f"DQ CHECK FAILED [{check_name}] → {count} violating rows")
    print(f"DQ CHECK PASSED [{check_name}]")


In [0]:
#Sanity check — row count (Gold vs Silver)

silver_count = spark.sql("""
    SELECT COUNT(*) FROM silver.stock_prices_cleaned
""").collect()[0][0]

gold_count = spark.sql("""
    SELECT COUNT(*) FROM gold.fact_stock_prices
""").collect()[0][0]

if gold_count > silver_count:
    raise Exception(
        f"DQ CHECK FAILED [row_count_sanity] → gold ({gold_count}) > silver ({silver_count})"
    )

print("DQ CHECK PASSED [row_count_sanity]")


In [0]:
#Null check — critical keys (FACT)

assert_zero(
    """
    SELECT COUNT(*)
    FROM gold.fact_stock_prices
    WHERE stock_key IS NULL
       OR date_key IS NULL
    """,
    "null_keys_in_fact"
)

#Duplicate grain check (FACT)

assert_zero(
    """
    SELECT COUNT(*)
    FROM (
      SELECT stock_key, date_key, COUNT(*) cnt
      FROM gold.fact_stock_prices
      GROUP BY stock_key, date_key
      HAVING COUNT(*) > 1
    )
    """,
    "duplicate_fact_grain"
)


#Foreign key integrity — stock_key

assert_zero(
    """
    SELECT COUNT(*)
    FROM gold.fact_stock_prices f
    LEFT JOIN gold.dim_stock d
      ON f.stock_key = d.stock_key
    WHERE d.stock_key IS NULL
    """,
    "fk_violation_stock_key"
)


#Foreign key integrity — date_key

assert_zero(
    """
    SELECT COUNT(*)
    FROM gold.fact_stock_prices f
    LEFT JOIN gold.dim_date d
      ON f.date_key = d.date_key
    WHERE d.date_key IS NULL
    """,
    "fk_violation_date_key"
)


print("ALL GOLD DATA QUALITY CHECKS PASSED")
